# Exploratory Data Analysis (EDA)
## Challenge #165 – Anatomic Structures Segmentation with Missing Labels (Raidium)

## Table of Contents
1. [Dataset Overview](#dataset-overview)
2. [Handling Missing Values](#handling-missing-values)
3. [Feature Distributions](#feature-distributions)
4. [Possible Biases](#possible-biases)
5. [Correlations](#correlations)


In [ ]:
# Import necessary libraries
import json
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import matplotlib.colors as mcolors
import seaborn as sns
from collections import Counter

sns.set_style('whitegrid')

# ── Paths (adjust to your local data folder) ──────────────────────────
TRAIN_INPUT_CSV = 'x_train.csv'
TRAIN_LABEL_CSV = 'y_train.csv'
ANNOTATED_JSON  = 'annotated_labels.json'
IMG_SIZE        = 256


## 1. Dataset Overview

**Source:** ENS Challenge Data #165 – Raidium.  
The training set contains **2 000 grayscale CT slices** (256×256 px), split into:
- **800 partially annotated** images with segmentation masks (structure IDs 1–23, 0 = background)
- **1 200 unannotated** images (masks filled with zeros)

Both `x_train` (images) and `y_train` (labels) are stored as transposed-flattened CSVs with shape `(65 536 pixels × N images)`.  
The `annotated_labels.json` lists which organ IDs are confirmed present per scan.


In [ ]:
# ── Load images ────────────────────────────────────────────────────────
x_raw    = pd.read_csv(TRAIN_INPUT_CSV, index_col=0, header=0)
x_images = x_raw.T.values.reshape((-1, IMG_SIZE, IMG_SIZE)).astype(np.float32)

# ── Load labels ────────────────────────────────────────────────────────
y_raw    = pd.read_csv(TRAIN_LABEL_CSV, index_col=0, header=0)
y_labels = y_raw.T.values.reshape((-1, IMG_SIZE, IMG_SIZE)).astype(np.uint8)

# ── Load annotation metadata ───────────────────────────────────────────
with open(ANNOTATED_JSON) as f:
    annotated = json.load(f)   # {image_id: [list of organ ids]}

N = len(x_images)
annotated_mask = np.array([y_labels[i].max() > 0 for i in range(N)])
n_annotated    = annotated_mask.sum()

print(f"Number of samples  (images) : {N}")
print(f"Number of features (pixels) : {IMG_SIZE}×{IMG_SIZE} = {IMG_SIZE**2}")
print(f"  └─ Annotated masks        : {n_annotated}")
print(f"  └─ Zero-label (unlabelled): {N - n_annotated}")
print(f"\nUnique structure IDs in labels: {np.unique(y_labels).tolist()}")
print(f"Image pixel range  : [{x_images.min():.2f}, {x_images.max():.2f}]")
print(f"\nExample data (first 5 rows/cols of x_raw):")
x_raw.iloc[:5, :5]


In [ ]:
# ── Visualise 6 sample slices with their masks ─────────────────────────
ann_images = x_images[annotated_mask]
ann_labels = y_labels[annotated_mask]

sample_idx = np.linspace(0, len(ann_images) - 1, 6, dtype=int)
cmap_mask  = plt.cm.get_cmap('tab20', 24)

fig, axes = plt.subplots(2, 6, figsize=(18, 6))
fig.suptitle('Sample CT Slices and Segmentation Masks', fontsize=13, fontweight='bold')

for col, idx in enumerate(sample_idx):
    axes[0, col].imshow(ann_images[idx], cmap='gray')
    axes[0, col].set_title(f'Slice {idx}', fontsize=9)
    axes[0, col].axis('off')
    axes[1, col].imshow(ann_labels[idx], cmap=cmap_mask, vmin=0, vmax=23)
    axes[1, col].axis('off')

axes[0, 0].set_ylabel('CT slice', fontsize=9)
axes[1, 0].set_ylabel('Seg mask', fontsize=9)
plt.tight_layout()
plt.show()


## 2. Handling Missing Values

The dataset has **two types of "missingness"**:

| Type | Meaning | Handling |
|---|---|---|
| CSV NaN values | Corrupt/missing pixel entries | Check & impute with 0 |
| All-zero label masks (1 200 images) | Intentionally unannotated — not missing data, but an unsupervised training signal | Do **not** impute; use for semi-supervised learning |

For pixel intensities there should be **no NaN values** after the challenge's official pre-processing.  
For the label CSV, zero is a valid value (background), so the only concern is true NaN entries.


In [ ]:
# ── Check for NaN values in the raw CSVs ──────────────────────────────
x_nan_count = x_raw.isnull().sum().sum()
y_nan_count = y_raw.isnull().sum().sum()

print(f"NaN pixels in x_train : {x_nan_count}")
print(f"NaN pixels in y_train : {y_nan_count}")

if x_nan_count > 0:
    print("  → Filling missing image pixels with 0 (air/background in HU)")
    x_raw.fillna(0, inplace=True)
    x_images = x_raw.T.values.reshape((-1, IMG_SIZE, IMG_SIZE)).astype(np.float32)

if y_nan_count > 0:
    print("  → Filling missing label pixels with 0 (background)")
    y_raw.fillna(0, inplace=True)
    y_labels = y_raw.T.values.reshape((-1, IMG_SIZE, IMG_SIZE)).astype(np.uint8)

# ── Intentionally unannotated split (not NaN, but label == 0 everywhere)
print(f"\nImages with all-zero masks (unannotated set) : {(~annotated_mask).sum()}")
print("  → Retained as unlabelled training data for semi-supervised learning.")


## 3. Feature Distributions

Key distributions to examine:
- **Pixel intensity** – CT Hounsfield-like values; indicates tissue types (air ≈ −1000 HU, fat ≈ −100, water/soft-tissue ≈ 0–80, bone ≈ 400+)
- **Structure pixel area** – how many pixels each organ occupies (class imbalance proxy)
- **Number of structures per slice** – slice diversity
- **Foreground coverage** – fraction of each image that is labelled


In [ ]:
# ── 3a. Global pixel intensity distribution ────────────────────────────
fig, axes = plt.subplots(1, 2, figsize=(14, 4))

flat_pixels = x_images.ravel()
axes[0].hist(flat_pixels, bins=200, color='steelblue', log=True, edgecolor='none')
axes[0].set_xlabel('Pixel Intensity')
axes[0].set_ylabel('Count (log scale)')
axes[0].set_title('Global Pixel Intensity Distribution')

# Per-image mean intensity
img_means = x_images.mean(axis=(1, 2))
axes[1].hist(img_means, bins=50, color='darkorange', edgecolor='none')
axes[1].set_xlabel('Mean pixel intensity per image')
axes[1].set_ylabel('# Images')
axes[1].set_title('Per-Image Mean Intensity')

plt.tight_layout()
plt.show()

p1, p5, p50, p95, p99 = np.percentile(flat_pixels, [1, 5, 50, 95, 99])
print(f"Intensity percentiles — P1:{p1:.1f}  P5:{p5:.1f}  P50:{p50:.1f}  P95:{p95:.1f}  P99:{p99:.1f}")


In [ ]:
# ── 3b. Per-structure pixel area distribution ──────────────────────────
unique_ids, pixel_counts = np.unique(ann_labels, return_counts=True)
struct_counts = {int(k): int(v) for k, v in zip(unique_ids, pixel_counts) if k != 0}

ids   = sorted(struct_counts.keys())
areas = [struct_counts[i] for i in ids]

fig, ax = plt.subplots(figsize=(12, 4))
bars = ax.bar(ids, areas, color='teal', edgecolor='white', linewidth=0.5)
ax.set_yscale('log')
ax.set_xlabel('Structure ID')
ax.set_ylabel('Total pixel area (log scale)')
ax.set_title('Pixel Area per Structure (Annotated Images Only)')
ax.set_xticks(ids)
plt.tight_layout()
plt.show()

print("Structure pixel areas (sorted by area, descending):")
for sid, area in sorted(struct_counts.items(), key=lambda x: -x[1]):
    print(f"  Structure {sid:>2}: {area:>10,} px")


In [ ]:
# ── 3c. Number of structures per annotated slice ───────────────────────
structs_per_slice = [len(np.unique(ann_labels[i])) - 1 for i in range(len(ann_labels))]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))

axes[0].hist(structs_per_slice, bins=range(0, max(structs_per_slice)+2),
             color='mediumvioletred', edgecolor='white', align='left')
axes[0].set_xlabel('# Structures in slice')
axes[0].set_ylabel('# Slices')
axes[0].set_title('Structures per Annotated Slice')

# Foreground (non-background) pixel ratio
fg_ratios = [(y > 0).mean() for y in ann_labels]
axes[1].hist(fg_ratios, bins=30, color='seagreen', edgecolor='white')
axes[1].set_xlabel('Foreground pixel ratio')
axes[1].set_ylabel('# Slices')
axes[1].set_title('Foreground Coverage per Annotated Slice')

plt.tight_layout()
plt.show()

print(f"Structures per slice — min:{min(structs_per_slice)}  max:{max(structs_per_slice)}"
      f"  mean:{np.mean(structs_per_slice):.1f}  median:{np.median(structs_per_slice):.1f}")
print(f"Foreground ratio    — min:{min(fg_ratios):.3f}  max:{max(fg_ratios):.3f}"
      f"  mean:{np.mean(fg_ratios):.3f}")


## 4. Possible Biases

The main bias risks in this challenge are:

| Bias type | Description |
|---|---|
| **Class imbalance** | Dominant structures (e.g. vertebrae, muscles) may have orders-of-magnitude more pixels than rare ones (e.g. tumours, small vessels) → DICE will be pulled by large structures |
| **Partial-label bias** | Some organs are never annotated together → the model may learn spurious negative correlations |
| **Anatomical region bias** | If annotated scans skew toward a specific body region (abdominal vs. thoracic), the model will generalise poorly to other regions |
| **Annotation frequency bias** | Organs annotated in few scans contribute disproportionately little gradient signal |


In [ ]:
# ── 4a. Class imbalance: pixel-area ratio between largest & smallest structure
max_area = max(struct_counts.values())
min_area = min(struct_counts.values())
print(f"Largest structure pixel area  : {max_area:,}")
print(f"Smallest structure pixel area : {min_area:,}")
print(f"Imbalance ratio               : {max_area / min_area:.1f}×")

# ── 4b. Organ annotation frequency (from annotated_labels.json) ────────
organ_freq = Counter()
for organs in annotated.values():
    organ_freq.update(organs)

organ_ids   = [k for k, _ in organ_freq.most_common()]
organ_freqs = [v for _, v in organ_freq.most_common()]

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(range(len(organ_ids)), organ_freqs, color='salmon', edgecolor='white')
ax.set_xticks(range(len(organ_ids)))
ax.set_xticklabels(organ_ids)
ax.axhline(n_annotated, color='crimson', linestyle='--', linewidth=1,
           label=f'# annotated images ({n_annotated})')
ax.set_xlabel('Organ ID')
ax.set_ylabel('# Scans where organ is annotated')
ax.set_title('Organ Annotation Frequency (annotated_labels.json) — Annotation Bias')
ax.legend()
plt.tight_layout()
plt.show()

print("\nOrgans annotated in fewer than 10% of annotated images:")
threshold = 0.1 * n_annotated
for oid, freq in organ_freq.items():
    if freq < threshold:
        print(f"  Organ {oid}: {freq} scans ({100*freq/n_annotated:.1f}%)")


In [ ]:
# ── 4c. Co-occurrence matrix: which organs appear together ─────────────
all_organ_ids = sorted(organ_freq.keys())
n_organs      = len(all_organ_ids)
id_to_idx     = {oid: i for i, oid in enumerate(all_organ_ids)}

cooccurrence = np.zeros((n_organs, n_organs), dtype=int)
for organs in annotated.values():
    for a in organs:
        for b in organs:
            if a in id_to_idx and b in id_to_idx:
                cooccurrence[id_to_idx[a], id_to_idx[b]] += 1

fig, ax = plt.subplots(figsize=(10, 8))
sns.heatmap(cooccurrence, xticklabels=all_organ_ids, yticklabels=all_organ_ids,
            cmap='YlOrRd', ax=ax, linewidths=0.3)
ax.set_title('Organ Co-occurrence in Annotated Scans\n(Reveals partial-label groupings / biases)')
ax.set_xlabel('Organ ID')
ax.set_ylabel('Organ ID')
plt.tight_layout()
plt.show()


## 5. Correlations

Since the input is image data (not tabular), classical feature–feature correlation matrices do not directly apply.  
Instead we explore:
- **Mean image intensity vs. foreground coverage** — does overall brightness predict how many structures are visible?
- **Structure size vs. structure ID** — are larger organs consistently assigned lower or higher IDs?
- **# structures vs. foreground ratio** — more organs ↔ more covered pixels?


In [ ]:
# ── 5a. Build a per-image summary dataframe ────────────────────────────
ann_img_means  = ann_images.mean(axis=(1, 2))
ann_img_stds   = ann_images.std(axis=(1, 2))

summary_df = pd.DataFrame({
    'mean_intensity' : ann_img_means,
    'std_intensity'  : ann_img_stds,
    'fg_ratio'       : fg_ratios,
    'n_structures'   : structs_per_slice,
})

print(summary_df.describe().round(3))


In [ ]:
# ── 5b. Correlation heatmap of per-image statistics ────────────────────
fig, axes = plt.subplots(1, 2, figsize=(13, 5))

corr = summary_df.corr()
sns.heatmap(corr, annot=True, fmt='.2f', cmap='coolwarm',
            vmin=-1, vmax=1, ax=axes[0], linewidths=0.5)
axes[0].set_title('Per-Image Feature Correlations')

# ── 5c. Scatter: n_structures vs fg_ratio ──────────────────────────────
axes[1].scatter(summary_df['n_structures'], summary_df['fg_ratio'],
                alpha=0.4, color='royalblue', edgecolors='none', s=20)
axes[1].set_xlabel('# Structures in slice')
axes[1].set_ylabel('Foreground pixel ratio')
axes[1].set_title('# Structures vs Foreground Coverage')

r = summary_df['n_structures'].corr(summary_df['fg_ratio'])
axes[1].annotate(f'r = {r:.2f}', xy=(0.05, 0.92), xycoords='axes fraction', fontsize=11)

plt.tight_layout()
plt.show()


In [ ]:
# ── 5d. Per-structure mean intensity (fg pixels only) ──────────────────
struct_mean_intensities = {}
for sid in ids:
    mask = ann_labels == sid        # bool mask across all annotated images
    if mask.any():
        struct_mean_intensities[sid] = float(ann_images[mask].mean())

fig, ax = plt.subplots(figsize=(12, 4))
ax.bar(struct_mean_intensities.keys(),
       struct_mean_intensities.values(),
       color='mediumpurple', edgecolor='white')
ax.set_xlabel('Structure ID')
ax.set_ylabel('Mean intensity of structure pixels')
ax.set_title('Per-Structure Mean Pixel Intensity\n(Encodes HU tissue type — useful for Hounsfield windowing)')
ax.set_xticks(list(struct_mean_intensities.keys()))
plt.tight_layout()
plt.show()

print("Per-structure mean intensities:")
for sid, mi in sorted(struct_mean_intensities.items(), key=lambda x: x[1]):
    print(f"  Structure {sid:>2}: {mi:>8.2f}")
